# Model V4 — XGBoost (68 features, Optuna-tuned)

XGBoost trained directly on the full **V3.1 15-min feature set** (68 features) —
one model, no subset comparison.

Data: `data/convertData/V3.1_15min_features.csv` (105,216 rows, 68 features).

**Loss function:** `reg:absoluteerror` (MAE). `regression_l1` is just a
deprecated alias of the same objective, so we use the canonical name.

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
import optuna
import joblib
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)


import sklearn
sklearn.set_config(display='text')

e:\Github\nordpool_electricity_price_prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [2]:
df = pd.read_csv('../data/convertData/V3.1_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)

features = [c for c in df.columns if c not in ('datetime', 'price')]
print('Shape:', df.shape)
print(f'Features: {len(features)}')
print('Date range:', df['datetime'].iloc[0], '->', df['datetime'].iloc[-1])

Shape: (105216, 70)
Features: 68
Date range: 2023-01-01 00:00:00+02:00 -> 2025-12-31 23:45:00+02:00


## 2. Train / Test Split

Chronological 80/20 — no shuffle. Test set is held out completely.

In [3]:
X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
train_end = n - int(n * 0.20)

X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]

print(f'Train : {X_train.shape}  ({X_train.shape[1]} features)')
print(f'Test  : {X_test.shape}')
print(f'Test period: {df["datetime"].iloc[train_end]} -> {df["datetime"].iloc[-1]}')

Train : (84173, 68)  (68 features)
Test  : (21043, 68)
Test period: 2025-05-26 20:15:00+03:00 -> 2025-12-31 23:45:00+02:00


## 3. Optuna Hyperparameter Search — TimeSeriesSplit CV

30 trials × 5-fold TimeSeriesSplit (no shuffling → no lookahead between folds).
Objective = average MAE across all folds.

In [4]:
tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'objective':         'reg:absoluteerror',
        'n_estimators':      2000,
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 50),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.001, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'random_state':      42,
        'verbosity':         0,
    }
    fold_maes = []
    for train_idx, val_idx in tscv.split(X_train):
        m = XGBRegressor(**params)
        m.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        fold_maes.append(mean_absolute_error(y_train.iloc[val_idx], m.predict(X_train.iloc[val_idx])))
    return np.mean(fold_maes)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest CV MAE : {study.best_value:.4f}')
print(f'Best params : {study.best_params}')

Best trial: 13. Best value: 2.90532: 100%|██████████| 30/30 [1:04:07<00:00, 128.26s/it]


Best CV MAE : 2.9053
Best params : {'learning_rate': 0.013933954180601458, 'max_depth': 12, 'min_child_weight': 42, 'subsample': 0.8853027013949373, 'colsample_bytree': 0.9926248953716896, 'reg_lambda': 0.005317607963898903, 'reg_alpha': 0.9729369493638725}


## 4. Train Final Model with Best Parameters

Train on the full training split with Optuna's best hyperparameters
(`study.best_params`). `n_estimators=2000` keeps the saved pkl small.

In [5]:
best = study.best_params
best.update({
    'objective':    'reg:absoluteerror',
    'n_estimators': 2000,
    'random_state': 42,
    'verbosity':    0,
})

model_v4 = XGBRegressor(**best)
model_v4.fit(X_train, y_train)
print('Training complete.')
print(f'Features: {X_train.shape[1]}')

Training complete.
Features: 68


## 5. Evaluate on Test Set

In [6]:
y_pred = model_v4.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R2   : {r2:.4f}')

MAE  : 2.7020
RMSE : 8.0376
R2   : 0.9730


## 6. Save

In [7]:
save_dir = Path('../models/saved')
save_dir.mkdir(exist_ok=True)

joblib.dump({
    'model':        model_v4,
    'feature_cols': X_train.columns.tolist(),
    'step_min':     15,
}, save_dir / 'xgboost_v4.pkl')

print('Saved -> models/saved/xgboost_v4.pkl')
print(f'Features ({X_train.shape[1]})')

Saved -> models/saved/xgboost_v4.pkl
Features (68)
